"""
This code is provided as supplemental material to the publication "Machine learning for small data sets: an exemplary study on the classification of highly complex surface micromorphologies" 
by M. Henkel, M. Sprenger, and O. Lieleg submitted to Materials Today Advances on October 17th, 2025.

"""

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import logging
import os
import yaml
from tqdm import tqdm
from PIL import Image
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_recall_fscore_support, accuracy_score, classification_report
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
class Config:
    """Configuration class to store hyperparameters"""
    def __init__(self, config_path='config_FSL.yaml'):
        #with command: automatically opens and close file 
        with open(config_path, 'r') as f:
            # .safe_load creates dictionary of f
            config = yaml.safe_load(f)
            
        self.include_NOD = config.get('include_NOD')
        self.num_epochs = config.get('num_epochs')
        self.batch_size = config.get('batch_size')
        self.learning_rate_feature_extractor = config.get('learning_rate_feature_extractor')
        self.learning_rate_classifier = config.get('learning_rate_classifier')
        self.num_classes = config.get('num_classes')
        self.image_size = config.get('image_size')
        self.train_ratio = config.get('train_ratio')
        self.val_ratio = config.get('val_ratio')
        self.samples_per_class = config.get('samples_per_class')  
        self.patience = config.get('patience')
        self.dropout_rate_FC = config.get('dropout_rate_FC')
        self.dropout_rate_CONV = config.get('dropout_rate_CONV')
        self.kernel_size = config.get('kernel_size')
        self.data_augmentation = config.get('data_augmentation')
        self.multiply_training_data = config.get('multiply_training_data')
        self.Freeze = config.get("Freeze")
        self.unfrozen_layers = config.get('unfrozen_layers')
        self.treshold_ABR = config.get("treshold_ABR")
        self.treshold_ADH = config.get("treshold_ADH")
        self.treshold_ERO = config.get("treshold_ERO")
        self.treshold_NOD = config.get("treshold_NOD")

In [ ]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()
        
        padding = (Config().kernel_size - 1) // 2
    
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size= Config().kernel_size, padding=padding),    
            nn.BatchNorm2d(16),                             
            nn.ReLU(),                                      
            nn.MaxPool2d(2, 2),                             
            nn.Dropout2d(Config().dropout_rate_CONV)                              
        )
        
        self.conv2 = nn.Sequential(
            nn.Conv2d(16, 32, kernel_size= Config().kernel_size, padding=padding),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(Config().dropout_rate_CONV)
        )
        
        self.conv3 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size= Config().kernel_size, padding=padding),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(Config().dropout_rate_CONV)
        )
        
        self.gap = nn.AdaptiveAvgPool2d(1)  
        
        self.fc1 = nn.Sequential(
            nn.Linear(64, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(Config().dropout_rate_FC)
        )

        self.fc2 = nn.Sequential(
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(Config().dropout_rate_FC)
        )

        self.fc3 = nn.Linear(128, Config().num_classes)
        
        
    def forward(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.conv3(x)
        x = self.gap(x)             
        x = x.view(x.size(0), -1)    
        x = self.fc1(x)                 
        x = self.fc2(x)
        features = x
        x = self.fc3(x)
        
        return x, features

In [ ]:
class FSLModel(nn.Module):
    def __init__(self, feature_extractor_state, num_features=128, num_classes=Config().num_classes, Freeze=Config().Freeze):
        super(FSLModel, self).__init__()
        
        # Create new ConvNet and load state
        self.feature_extractor = ConvNet()
        
        # Load pre-trained parameters
        self.feature_extractor.load_state_dict(feature_extractor_state, strict=False)
        
        # Freeze all layers initially
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
            
        # Unfreeze specific layers 
        if not Freeze:
            total_params = 0
            unfrozen_params = 0

            print("\nUnfrozen layers:")
            for name, param in self.feature_extractor.named_parameters():
                if any(layer in name for layer in Config().unfrozen_layers):
                    param.requires_grad = True
                    print(f"• {name}")
            
            # Then calculate and show parameter counts
            for name, param in self.feature_extractor.named_parameters():
                total_params += param.numel()
                if param.requires_grad:
                    unfrozen_params += param.numel()

            print(f"\nParameter summary:")
            print(f"Total parameters: {total_params:,}")
            print(f"Retrained parameters: {unfrozen_params:,}")
            print(f"Percentage retrained: {(unfrozen_params/total_params)*100:.2f}%\n")
                    
        # Create new classification layer  
        self.classifier = nn.Linear(num_features, num_classes)
        
    def forward(self, x):
        logits, features = self.feature_extractor(x)  
        return self.classifier(features)  

In [ ]:
class MultiLabelDataset(Dataset):
    def __init__(self, data_path, transform=None):
        self.data_path = data_path
        self.transform = transform
        
        # Modify base classes based on config
        if Config().include_NOD:
            self.base_classes = ['ABR', 'ADH', 'ERO', 'NOD']
        else:
            self.base_classes = ['ABR', 'ADH', 'ERO']
        
        # Create empty list to hold (image_path, multi_hot_label) tuples
        self.samples = []
        
        # Get all folders for damage classes
        class_folders = [d.name for d in os.scandir(data_path) if d.is_dir()]
        
        for folder in class_folders:
            if not Config().include_NOD and 'NOD' in folder:
                continue
                
            # Get multi-hot vector for this folder name
            multi_hot = self.parse_class_name(folder)
            folder_path = os.path.join(data_path, folder)
            
            # Fuse all images in this folder with their labels (Multi-hot vectors)
            image_files = [f for f in os.listdir(folder_path)
                         if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            for img_file in image_files:
                img_path = os.path.join(folder_path, img_file)
                self.samples.append((img_path, multi_hot))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        
        if self.transform:
            image = self.transform(image)
            
        return image, torch.tensor(label, dtype=torch.float32), img_path

    def parse_class_name(self, class_name):
        """Convert folder names like 'ABR_ADH' to multi-hot vectors"""
        individual_classes = class_name.split('_')
        
        multi_hot = []
        for base_class in self.base_classes:
            multi_hot.append(1 if base_class in individual_classes else 0)
        
        return multi_hot

In [ ]:
class DatasetManagerFSL:
    @staticmethod
    def get_transforms(training=False):
        if training:
            return transforms.Compose([
                transforms.CenterCrop((419, 559)),  # (height, width) (419, 559)
                transforms.Resize((Config().image_size, Config().image_size)),
                transforms.RandomAffine(
                    degrees=10,  
                    translate=(0.05, 0.05),         
                    scale=(0.95, 1.05),  
                    shear=None  
                ),
                transforms.RandomHorizontalFlip(p=0.3),
                transforms.RandomVerticalFlip(p=0.3),
                transforms.ToTensor(),
            ])
    
        else:
            return transforms.Compose([
                transforms.CenterCrop((419, 559)),  # (height, width) (419, 559)
                transforms.Resize((Config().image_size, Config().image_size)),
                transforms.ToTensor(),
            ])


    @staticmethod
    def load_data(data_path):
        # Load full dataset (triplet containing image, label, image_path)
        full_dataset = MultiLabelDataset(data_path=data_path)
        
        # Group samples by class combinations
        class_combinations = {}
        for idx in range(len(full_dataset)):
            _, label,_ = full_dataset[idx]
            label_tuple = tuple(label.numpy())
            if label_tuple not in class_combinations:
                class_combinations[label_tuple] = []
            class_combinations[label_tuple].append(idx)
        
        # Select training samples
        train_indices = []
        generator = torch.Generator().manual_seed(17)  
        for indices in class_combinations.values():
            indices_tensor = torch.tensor(indices)
            shuffled_indices = indices_tensor[torch.randperm(len(indices_tensor), generator=generator)]
            n_samples = Config().samples_per_class
            train_indices.extend(shuffled_indices[:n_samples].tolist())
        
        # Get remaining indices for validation and test
        remaining_indices = list(set(range(len(full_dataset))) - set(train_indices))
        
        # Split remaining data into validation and test
        val_size = int(Config().val_ratio * len(full_dataset))
        generator = torch.Generator().manual_seed(42)   
        remaining_indices = torch.tensor(remaining_indices)[torch.randperm(len(remaining_indices), generator=generator)]
        val_indices = remaining_indices[:val_size].tolist()
        test_indices = remaining_indices[val_size:].tolist()
        
        # Create datasets
        train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
        val_dataset = torch.utils.data.Subset(full_dataset, val_indices)
        test_dataset = torch.utils.data.Subset(full_dataset, test_indices)
        
        # Create duplicates for data augmentation
        if Config().data_augmentation:
            extended_indices = (
                # Part 1: Guarantee each image appears once
                list(range(len(train_dataset))) +
                
                # Part 2: Add additional random samples:
                # - len(train_dataset) = size of original training set
                # - multiply_training_data - 1 = number of additional copies needed
                # - torch.randint generates random indices between 0 and len(train_dataset)
                # - tolist() converts the tensor of random indices to a Python list
                torch.randint(0, len(train_dataset),
                            (len(train_dataset) * (Config().multiply_training_data - 1),)
                ).tolist()
            )
            
            # Create new training dataset with both original and augmented samples
            # Subset() creates a view of the dataset using the specified indices
            train_dataset = torch.utils.data.Subset(train_dataset, extended_indices)
        else:
            pass
        
        
        # Transform datasets
        train_dataset.dataset.transform = DatasetManagerFSL.get_transforms(training=True)
        val_dataset.dataset.transform = DatasetManagerFSL.get_transforms(training=False)
        test_dataset.dataset.transform = DatasetManagerFSL.get_transforms(training=False)

        # Create dataloaders
        train_loader = DataLoader(
            train_dataset,
            batch_size=Config().batch_size,
            shuffle=True,
            num_workers=0,
            drop_last=True
        )
        
        val_loader = DataLoader(
            val_dataset,
            batch_size=Config().batch_size,
            shuffle=False,
            num_workers=0,
            drop_last=True
        )
        
        test_loader = DataLoader(
            test_dataset,
            batch_size=Config().batch_size,
            shuffle=False,
            num_workers=0,
            drop_last=True
        )
        
        # Logging
        total_samples = len(train_loader.dataset) + len(val_loader.dataset) + len(test_loader.dataset)

        if Config().data_augmentation:
            original_train_size = len(train_loader.dataset) // Config().multiply_training_data
            logging.info(
                f'Dataset sizes (with data augmentation x{Config().multiply_training_data}):\n'
                f'Training:   {len(train_loader.dataset):5d} samples (original: {original_train_size} samples)\n'
                f'Validation: {len(val_loader.dataset):5d} samples\n'
                f'Test:       {len(test_loader.dataset):5d} samples\n'
                f'Total:      {total_samples:5d} samples'
            )
        else:
            logging.info(
                f'Dataset sizes (no data augmentation):\n'
                f'Training:   {len(train_loader.dataset):5d} samples\n'
                f'Validation: {len(val_loader.dataset):5d} samples\n'
                f'Test:       {len(test_loader.dataset):5d} samples\n'
                f'Total:      {total_samples:5d} samples'
            )
        
        return train_loader, val_loader, test_loader

In [ ]:
class EarlyStopping:
    def __init__(self, patience=Config().patience, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        
    def __call__(self, score):  
        if self.best_score is None:
            self.best_score = score
            logging.info(f"Early Stopping: Setting initial best score: {score:.2f}")
            return False
            
        if score <= (self.best_score + self.min_delta):
            self.counter += 1
            logging.info(f"Early Stopping: No improvement. Counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                logging.info("Early Stopping: Patience exceeded!")
        else:
            logging.info(f"Early Stopping: Improvement from {self.best_score:.2f} to {score:.2f}")
            self.best_score = score
            self.counter = 0
            
        return self.early_stop

In [ ]:
class Trainer:
    def __init__(self, model, device):
        self.model = model
        self.device = device
        
        self.criterion = nn.BCEWithLogitsLoss()
        self.early_stopping = EarlyStopping()
        
        # Create empty lists for different parameter groups
        feature_params = []
        classifier_params = []
        
        # Sort the parameters into their group
        for name, param in model.named_parameters():
            for item in Config().unfrozen_layers:
                if item in name:
                    feature_params.append(param)
            if 'classifier' in name:
                classifier_params.append(param)
                    
        # Create optimizer with different learning rates
        self.optimizer = torch.optim.AdamW([
            {
                'params': feature_params,    
                'lr': Config().learning_rate_feature_extractor,
                'weight_decay': 0.01
            },
            {
                'params': classifier_params, 
                'lr': Config().learning_rate_classifier,
                'weight_decay': 0.01
            }
        ])     

        self.scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            self.optimizer,
            mode='max',           
            factor=0.1,
            patience=3,            
            verbose=True,
            min_lr=1e-6 
        )
        
    def save_checkpoint(self, filename, epoch, loss, accuracy):
        """Save model checkpoint with all relevant training information"""
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'loss': loss,
            'accuracy': accuracy
        }
        torch.save(checkpoint, filename)
        logging.info(f'Checkpoint saved to {filename}')

    def train(self, train_loader, val_loader, num_epochs=Config().num_epochs):
        best_accuracy = 0
        save_dir = 'saved_FSL_models'
        os.makedirs(save_dir, exist_ok=True)
        learning_rates = []  # Track learning rates for both parameter groups
        
        for epoch in range(num_epochs):
            # Get current learning rates
            current_lrs = [group['lr'] for group in self.optimizer.param_groups]
            learning_rates.append(current_lrs)
            
            logging.info(f'\nEpoch {epoch+1}/{Config().num_epochs}')
            logging.info(f'Current LRs - Feature Extractor: {current_lrs[0]:.6f}, Classifier: {current_lrs[1]:.6f}')
            
            # Training Phase
            self.model.train()
            train_loss = 0
            
            for images, labels,_ in tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}'): # Progress bar entails training dataset
                images = images.to(self.device)                                               # Send images to CPU or GPU
                labels = labels.to(self.device)                                               # Send labels to CPU or GPU
                
                self.optimizer.zero_grad()                                                    # Zero gradients

                outputs = self.model(images)                                                  # Train and evaluate model
                loss = self.criterion(outputs, labels)
                loss.backward()
                self.optimizer.step()
                
                train_loss += loss.item()
            
            avg_train_loss = train_loss / len(train_loader)
            
            # Validation Phase
            self.model.eval()
            val_loss = 0
            correct_predictions = 0
            total_samples = 0
            
            # Retrive decison thresholds from config
            if Config().include_NOD:
                thresholds = torch.tensor([
                    Config().treshold_ABR, 
                    Config().treshold_ADH, 
                    Config().treshold_ERO,
                    Config().treshold_NOD
                ]).to(self.device)
            else:
                thresholds = torch.tensor([
                    Config().treshold_ABR, 
                    Config().treshold_ADH, 
                    Config().treshold_ERO
                ]).to(self.device)
            
            # Predict validation dataset
            with torch.no_grad():
                for images, labels,_ in val_loader:
                    images = images.to(self.device)
                    labels = labels.to(self.device)
                    
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                    val_loss += loss.item()
                    
                    sigmoid_outputs = torch.sigmoid(outputs)
                    predicted = (sigmoid_outputs > thresholds).float()
                    
                    for i in range(len(labels)):
                        if torch.equal(predicted[i], labels[i]):
                            correct_predictions += 1
                    total_samples += len(labels)
                    
            avg_val_loss = val_loss / len(val_loader)
            accuracy = 100 * correct_predictions / total_samples

    
            # Control learning rates
            self.scheduler.step(accuracy)
            new_lrs = [group['lr'] for group in self.optimizer.param_groups]
            if new_lrs != current_lrs:
                logging.info(f'Learning rates reduced to - Feature Extractor: {new_lrs[0]:.6f}, Classifier: {new_lrs[1]:.6f}')
                checkpoint_path = os.path.join(save_dir, 'best_fsl_model.pth')
                if os.path.exists(checkpoint_path):
                    checkpoint = torch.load(checkpoint_path, map_location=self.device)
                    self.model.load_state_dict(checkpoint['model_state_dict'])
                    logging.info('Loaded best model state from checkpoint file')
            
            # Save checkpoint if accuracy improved
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                checkpoint_path = os.path.join(save_dir, 'best_fsl_model.pth')
                self.save_checkpoint(checkpoint_path, epoch, avg_train_loss, accuracy)
                logging.info(f'Saved new best model with accuracy: {accuracy:.2f}%')
            
            # Log progress
            logging.info(
                f'Epoch [{epoch+1}/{num_epochs}] | '\
                f'Train Loss: {avg_train_loss:.4f} | '\
                f'Val Loss: {avg_val_loss:.4f} | '\
                f'Exact Match Accuracy: {accuracy:.2f}% | '\
                f'LR Feature Extractor: {current_lrs[0]:.6f} | '\
                f'LR Classifier: {current_lrs[1]:.6f}'\
            )
            
            # Early stopping check
            if self.early_stopping(accuracy):
                logging.info(f"Early stopping triggered at epoch {epoch+1}")
                if os.path.exists(checkpoint_path):
                        checkpoint = torch.load(checkpoint_path, map_location=self.device)
                        self.model.load_state_dict(checkpoint['model_state_dict'])
                        logging.info('Loaded best model state from checkpoint file')
                break
        
        # Plot learning rates
        plt.figure(figsize=(10, 5))
        feature_rates = [lrs[0] for lrs in learning_rates]
        classifier_rates = [lrs[1] for lrs in learning_rates]
        plt.plot(feature_rates, label='Feature Extractor LR')
        plt.plot(classifier_rates, label='Classifier LR')
        plt.yscale('log')
        plt.xlabel('Epoch')
        plt.ylabel('Learning Rate')
        plt.title('Learning Rate Schedule')
        plt.legend()
        save_dir = 'visualizations'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig('visualizations/learning_rates.png')
        plt.close()
        
        return self.model

In [ ]:
def get_device():
        if torch.backends.mps.is_available():
            device = torch.device("mps")
            logging.info(f'Using device: MPS (Apple Silicon GPU)')
        elif torch.cuda.is_available():
            device = torch.device("cuda")
            logging.info(f'Using device: CUDA GPU')
        else:
            device = torch.device("cpu")
            logging.info(f'Using device: CPU')
        return device

In [ ]:
def main():
    logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
    device = get_device()
    
    try:
        # Create save directory
        save_dir = 'saved_FSL_models'
        os.makedirs(save_dir, exist_ok=True)
        
        # 1. Load the data
        logging.info('Loading and preparing datasets...')
        train_loader, val_loader, test_loader = DatasetManagerFSL.load_data('FSL_IMG_Data')
        
        # 2. Load the feature extractor state
        logging.info('Loading feature extractor model...')
        feature_extractor_state = torch.load(
            'saved_models/feature_extractor.pth',
            map_location=device,
            weights_only=True
        )
        
        # 3. Create FSL model
        logging.info('Initializing FSL model...')
        model = FSLModel(feature_extractor_state).to(device)
        
        # 4. Create trainer and train model
        logging.info('Starting model training...')
        trainer = Trainer(model, device)
        trained_model = trainer.train(train_loader, val_loader)

        # Set thresholds for evaluation
        if Config().include_NOD:
            thresholds = torch.tensor([
                Config().treshold_ABR, 
                Config().treshold_ADH, 
                Config().treshold_ERO,
                Config().treshold_NOD
            ]).to(device)
        else:
            thresholds = torch.tensor([
                Config().treshold_ABR, 
                Config().treshold_ADH, 
                Config().treshold_ERO
            ]).to(device)
            
        # 6. Test the model
        logging.info('Evaluating model on test set...')
        trained_model.eval()
        
        # Dynamically set damage types for the evaluation of the test dataset
        damage_types = ['ABR', 'ADH', 'ERO']
        if Config().include_NOD:
            damage_types.append('NOD')

        # For storing results
        all_predictions = []
        all_labels = []
        all_filenames = []
        
        # Testing the model
        with torch.no_grad():
            for images, labels, paths in test_loader:
                images = images.to(device)
                labels = labels.to(device)
                all_filenames.extend(paths)
                
                outputs = trained_model(images)                 
                sigmoid_outputs = torch.sigmoid(outputs)
                predicted = torch.zeros_like(sigmoid_outputs)

                # Finding the two likeliest classes
                for idx in range(sigmoid_outputs.shape[0]):
                    scores = sigmoid_outputs[idx]
                    top2 = torch.topk(scores, 2)
                    top2_indices = top2.indices
                    top2_scores = top2.values
                    
                    # Limiting test output to physically meaningful outputs, e.g., preventing outputs like: ABR+NOD 
                    if Config().include_NOD:
                        nod_idx = damage_types.index('NOD')
                        # If NOD is in top two
                        if nod_idx in top2_indices:
                            # Only select the higher score (NOD or other)
                            max_idx = top2_indices[0] if top2_scores[0] >= top2_scores[1] else top2_indices[1]
                            if scores[max_idx] > thresholds[max_idx]:
                                predicted[idx] = torch.zeros_like(predicted[idx])
                                predicted[idx, max_idx] = 1
                            else:
                                # If neither above threshold, still select the highest
                                predicted[idx] = torch.zeros_like(predicted[idx])
                                predicted[idx, max_idx] = 1
                        else:
                            for i in top2_indices:
                                if scores[i] > thresholds[i]:
                                    predicted[idx, i] = 1
                            # If none above threshold (predicted: [0,0,0]), select the highest one
                            if predicted[idx].sum() == 0:
                                predicted[idx, top2_indices[0]] = 1
                    else:
                        # No NOD: standard logic
                        for i in top2_indices:
                            if scores[i] > thresholds[i]:
                                predicted[idx, i] = 1
                        if predicted[idx].sum() == 0:
                            predicted[idx, top2_indices[0]] = 1           

                
                # Convert vectors to class combination strings
                for pred, label in zip(predicted, labels):
                    pred_classes = []
                    true_classes = []
                    #Loop through label and prediction vector to map class allocations to class names
                    for i, (p, l) in enumerate(zip(pred, label)):
                        if p == 1:
                            pred_classes.append(damage_types[i])
                        if l == 1:
                            true_classes.append(damage_types[i])
                    pred_str = '+'.join(sorted(pred_classes)) if pred_classes else 'NONE'
                    true_str = '+'.join(sorted(true_classes)) if true_classes else 'NONE'
                    
                    all_predictions.append(pred_str)
                    all_labels.append(true_str)

        # Save predictions to CSV
        df_predictions = pd.DataFrame({
            'image': all_filenames,
            'true_label': all_labels,
            'predicted_label': all_predictions
            })
        df_predictions.to_csv('predictions_FSL.csv', index=False)
        logging.info("All predictions saved as predictions_FSL.csv.")
        
        # Calculate metrics using sklearn
        unique_combinations = sorted(list(set(all_predictions + all_labels)))
        report = classification_report(
            all_labels,
            all_predictions,
            labels=unique_combinations,
            zero_division=0,
            output_dict=True
        )
        
        # Print metrics table
        logging.info('\nMetrics Summary:')
        logging.info('\n{:<15} {:<12} {:<12} {:<12} {:<12}'.format(
            'Class', 'Precision', 'Recall', 'F1-Score', 'Support'))
        logging.info('-' * 63)

        # Print per-class metrics
        for combo in unique_combinations:
            metrics = report[combo]
            logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
                combo,
                metrics['precision'],
                metrics['recall'],
                metrics['f1-score'],
                metrics['support']
            ))
        
        # Print average metrics
        logging.info('-' * 63)
        logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
            'Macro Avg.',
            report['macro avg']['precision'],
            report['macro avg']['recall'],
            report['macro avg']['f1-score'],
            report['macro avg']['support']
        ))
        logging.info('{:<15} {:<12.2f} {:<12.2f} {:<12.2f} {:<12.0f}'.format(
            'Weighted Avg.',
            report['weighted avg']['precision'],
            report['weighted avg']['recall'],
            report['weighted avg']['f1-score'],
            report['weighted avg']['support']
        ))
        
        # Create confusion matrix
        plt.figure(figsize=(12, 10))
        cm = confusion_matrix(all_labels, all_predictions, labels=unique_combinations)
        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm,
            display_labels=unique_combinations
        )
        disp.plot(xticks_rotation=90)
        plt.title('Confusion Matrix')
        plt.tight_layout()
        
        # Save with different name based on NOD inclusion
        matrix_filename = 'confusion_matrix_FSL.png' 
        save_dir = 'visualizations'
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(f'visualizations/{matrix_filename}')
        plt.close()
        
    except Exception as e:
        logging.error(f'An error occurred: {str(e)}')
        import traceback
        logging.error(traceback.format_exc())
    finally:
        torch.cuda.empty_cache()

if __name__ == '__main__':
    main()